In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r'C:\Infoct_project 1 sample practice\dataset\predictive_maintenance.csv')
print("Dataset loaded!")
print(df.shape)

Dataset loaded!
(10000, 10)


In [3]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Week 2 external context
np.random.seed(42)
df['Ambient_temperature'] = np.random.normal(loc=295, scale=3, size=len(df))
df['Load_density'] = np.random.uniform(low=0.3, high=1.0, size=len(df))

# Week 1 engineered features
df["temp_difference"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"]
df["tool_wear_rate"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"]
df["heat_stress_index"] = df["Air temperature [K]"] * df["Tool wear [min]"]
df["wear_per_rotation"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"] * 1000

# Week 3 Day 1 new features
df["thermal_stress"] = df["temp_difference"] * df["tool_wear_rate"]
df["power_per_temp"] = df["power"] / df["Air temperature [K]"]

print("All features recreated!")
print(df.shape)

All features recreated!
(10000, 19)


In [7]:
# Final 12 features selected from Day 2
final_features = ['Air temperature [K]', 'Process temperature [K]','Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'temp_difference', 'power', 'tool_wear_rate',
'heat_stress_index', 'wear_per_rotation','thermal_stress', 'power_per_temp']
X = df[final_features]
Y = df['Target']
print(f"Features shape: {X.shape}")
print(f"Target shape: {Y.shape}")
print(f"Failure rate: {Y.mean()*100:.2f}%")

Features shape: (10000, 12)
Target shape: (10000,)
Failure rate: 3.39%


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")
print(f"Training failures: {Y_train.sum()}")
print(f"Testing failures: {Y_test.sum()}")

Training set: (8000, 12)
Testing set: (2000, 12)
Training failures: 271
Testing failures: 68


In [9]:
import lightgbm as lgb
from sklearn.metrics import f1_score, classification_report

# Train LightGBM model
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train, Y_train)

# Predictions
pred = model.predict(X_test)

# Score
score = f1_score(Y_test, pred, average='macro')
print(f"LightGBM Macro F1 Score: {score:.3f}")
print("\nClassification Report:")
print(classification_report(Y_test, pred))

[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000335 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2536
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM Macro F1 Score: 0.899

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.79      0.82      0.81        68

    accuracy                           0.99      2000
   macro avg       0.89      0.91      0.90      2000
weighted avg       0.99      0.99      0.99      2000



c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
print("=== Model Comparison ===")
print(f"Logistic Regression (Week 2) Macro F1: 0.693")
print(f"LightGBM (Week 3) Macro F1: {score:.3f}")
print(f"Improvement: {score - 0.693:.3f}")
print(f"\nLightGBM is {((score - 0.693) / 0.693 * 100):.1f}% better than Logistic Regression!")

=== Model Comparison ===
Logistic Regression (Week 2) Macro F1: 0.693
LightGBM (Week 3) Macro F1: 0.899
Improvement: 0.206

LightGBM is 29.8% better than Logistic Regression!
